# Satellite Orbit Explorer — ECI & ECEF Edition
**Built by Ehsan Gholami · Space Career Week 2026 · GNSS Academy**

This notebook is an educational two-body Keplerian orbit propagator. It calculates an approximate
Earth-centred inertial trajectory (**ECI-like**) and transforms it into Earth-centred, Earth-fixed
coordinates (**ECEF**) using a Greenwich sidereal rotation approximation.

## Workflow
1. Setup and configuration
2. Load the original GP JSON input data
3. Parse and validate orbital elements
4. Create a common UTC time grid
5. Propagate positions and velocities in ECI, then transform them to ECEF
6. Summarize the orbits and run numerical checks
7. Plot ECI trajectories, ground tracks, and orbital speeds
8. Export ECI/ECEF CSV files, KML files, plots, a report, and a ZIP archive

**Model limitations:** CelesTrak GP/OMM mean elements are intended for SGP4-compatible propagation.
Using them directly in a two-body Kepler model is an educational approximation, not an SGP4 replacement.
The ECI frame here is therefore labelled **ECI-like**. GMST uses UTC as an approximation to UT1;
precession, nutation, polar motion, J2, drag, and other perturbations are not modelled.

In [ ]:
# =====================================================================
# Stage 1: imports, constants, folders, and configuration
# =====================================================================
import csv
import json
import math
import zipfile
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta, timezone
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import to_hex

MU_EARTH_KM3_S2 = 398600.4418       # km^3/s^2
EARTH_RADIUS_KM = 6378.137          # spherical Earth radius, km
EARTH_ROTATION_RAD_S = 7.29211514670698e-5  # approximate Earth rotation rate

# In VS Code, open the project folder before running the notebook.
BASE_DIR = Path.cwd()
IN_DIR = BASE_DIR / "IN"
OUT_DIR = BASE_DIR / "OUT"
IN_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

STEP_SECONDS = 60                   # 1-minute sampling; change to 5 for finer output
DURATION_HOURS = 24
INPUT_ZIP_PATTERN = "orbit_workshop_inputs*.zip"

print(f"Project folder: {BASE_DIR}")
print(f"Input folder:   {IN_DIR}")
print(f"Output folder:  {OUT_DIR}")
print(f"Window: {DURATION_HOURS} h | step: {STEP_SECONDS} s")

## Stage 2: Load GP JSON and required map data

The notebook expects the original workshop input data, not the previously generated output ZIP.
Place either `orbit_workshop_inputs.zip` in the project folder, or place these files directly in `IN`:

- `orbital_elements.json` — the actual GP/OMM records
- `world_map.txt` — coastline segments used by the ground-track plot
- `data_manifest.json` — optional provenance information

The notebook stops with a clear error if the required source data is missing. It does not create
or substitute satellite records.

In [ ]:
# =====================================================================
# Stage 2: find and load the original input data
# =====================================================================
REQUIRED_INPUTS = ["orbital_elements.json", "world_map.txt"]
GP_FIELDS = [
    "OBJECT_NAME", "NORAD_CAT_ID", "EPOCH", "MEAN_MOTION",
    "ECCENTRICITY", "INCLINATION", "RA_OF_ASC_NODE",
    "ARG_OF_PERICENTER", "MEAN_ANOMALY"
]

def extract_input_zip():
    # Look only in the project folder and IN folder.
    candidates = sorted(BASE_DIR.glob(INPUT_ZIP_PATTERN)) + sorted(IN_DIR.glob(INPUT_ZIP_PATTERN))
    for archive in candidates:
        if zipfile.is_zipfile(archive):
            with zipfile.ZipFile(archive) as zf:
                names = set(zf.namelist())
                missing = [name for name in REQUIRED_INPUTS if name not in names]
                if missing:
                    continue
                for name in REQUIRED_INPUTS + ["data_manifest.json"]:
                    if name in names:
                        (IN_DIR / name).write_bytes(zf.read(name))
            print(f"Extracted source files from: {archive.name}")
            return archive
    return None

if not all((IN_DIR / name).is_file() for name in REQUIRED_INPUTS):
    extract_input_zip()

missing_files = [name for name in REQUIRED_INPUTS if not (IN_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(
        "Missing original input file(s): " + ", ".join(missing_files) +
        f". Put orbit_workshop_inputs.zip in {BASE_DIR}, or put the required files in {IN_DIR}. "
        "The supplied satellite_orbit_outputs.zip contains outputs only, not these source inputs."
    )

def read_gp_json(path):
    records = json.loads(Path(path).read_text(encoding="utf-8"))
    if not isinstance(records, list) or not records:
        raise ValueError(f"{path} must contain a non-empty JSON list of GP records.")
    for index, record in enumerate(records):
        missing = [field for field in GP_FIELDS if field not in record]
        if missing:
            raise ValueError(f"GP record {index} is missing required field(s): {missing}")
    return records

sats = read_gp_json(IN_DIR / "orbital_elements.json")
manifest_path = IN_DIR / "data_manifest.json"
if manifest_path.is_file():
    data_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    source_note = f"Workshop GP snapshot retrieved {data_manifest.get('retrieved_utc', 'date not recorded')}"
else:
    source_note = "GP input file supplied locally; provenance manifest not provided"

print(source_note)
print(f"Loaded {len(sats)} actual GP records:")
for sat in sats:
    print(f"  {sat['OBJECT_NAME']} | NORAD {sat['NORAD_CAT_ID']} | epoch {sat['EPOCH']}")

## Stage 3: Map GP fields to Keplerian model elements

Mean motion is converted from revolutions/day to radians/second, and the two-body semi-major axis
is estimated with Kepler's third law. The input elements are validated before propagation.

In [ ]:
# =====================================================================
# Stage 3: parse and validate the GP records
# =====================================================================
def parse_gp_epoch(value):
    # Accept ISO-8601 timestamps with Z, an explicit offset, or no timezone.
    dt = datetime.fromisoformat(value.replace("Z", "+00:00"))
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc)

def parse_gp_elements(record):
    numeric_fields = [
        "MEAN_MOTION", "ECCENTRICITY", "INCLINATION",
        "RA_OF_ASC_NODE", "ARG_OF_PERICENTER", "MEAN_ANOMALY"
    ]
    values = {key: float(record[key]) for key in numeric_fields}
    mean_motion = 2 * math.pi * values["MEAN_MOTION"] / 86400.0
    eccentricity = values["ECCENTRICITY"]

    if mean_motion <= 0:
        raise ValueError(f"{record['OBJECT_NAME']}: MEAN_MOTION must be positive.")
    if not 0 <= eccentricity < 1:
        raise ValueError(f"{record['OBJECT_NAME']}: this notebook requires 0 <= eccentricity < 1.")

    return {
        "name": str(record["OBJECT_NAME"]),
        "norad_id": int(record["NORAD_CAT_ID"]),
        "label": f"{record['OBJECT_NAME']} [{int(record['NORAD_CAT_ID'])}]",
        "epoch": parse_gp_epoch(str(record["EPOCH"])),
        "n_rad_s": mean_motion,
        "a_km": (MU_EARTH_KM3_S2 / mean_motion**2) ** (1/3),
        "e": eccentricity,
        "i_rad": math.radians(values["INCLINATION"]),
        "raan_rad": math.radians(values["RA_OF_ASC_NODE"]),
        "argp_rad": math.radians(values["ARG_OF_PERICENTER"]),
        "M0_rad": math.radians(values["MEAN_ANOMALY"]),
    }

def elements_table(items):
    lines = [
        f"{'Satellite':<24} {'NORAD':>7} {'Epoch UTC':<20} {'a [km]':>10} "
        f"{'e':>10} {'i [deg]':>9} {'RAAN':>9} {'argp':>9} {'M0':>9}"
    ]
    for el in items:
        angles = [math.degrees(el[k]) for k in ("i_rad", "raan_rad", "argp_rad", "M0_rad")]
        lines.append(
            f"{el['name']:<24} {el['norad_id']:>7} {el['epoch']:%Y-%m-%d %H:%M:%S} "
            f"{el['a_km']:>10.2f} {el['e']:>10.6f} " +
            " ".join(f"{angle:>9.3f}" for angle in angles)
        )
    return lines

elements = [parse_gp_elements(sat) for sat in sats]
print("\n".join(elements_table(elements)))

## Stage 4: Create a common UTC time grid

The grid starts at UTC midnight on the newest element epoch and runs for the configured duration.
Older records may therefore be propagated away from their own epoch; the printed offsets help
identify stale input elements.

In [ ]:
# =====================================================================
# Stage 4: create the shared UTC time grid
# =====================================================================
newest_epoch = max(el["epoch"] for el in elements)
start = newest_epoch.replace(hour=0, minute=0, second=0, microsecond=0)
end = start + timedelta(hours=DURATION_HOURS)
n_points = int((end - start).total_seconds()) // STEP_SECONDS + 1
times = [start + timedelta(seconds=k * STEP_SECONDS) for k in range(n_points)]

print(f"UTC window: {start:%Y-%m-%d %H:%M:%S} -> {end:%Y-%m-%d %H:%M:%S}")
print(f"Step: {STEP_SECONDS} s | points per satellite: {n_points}")
print(f"{'Satellite':<24} {'Start minus epoch [days]':>25}")
for el in elements:
    offset_days = (start - el["epoch"]).total_seconds() / 86400
    warning = "  <-- older than 7 days" if abs(offset_days) > 7 else ""
    print(f"{el['name']:<24} {offset_days:>25.2f}{warning}")

## Stage 5: Propagate in ECI-like coordinates and transform to ECEF

For each time, the notebook solves Kepler's equation, computes position and velocity in the orbital
plane, and rotates them into an Earth-centred inertial-like frame. It then rotates the position into
ECEF using GMST. ECEF velocity also accounts for the rotating coordinate frame.

Coordinate conventions:
- ECI-like position/velocity: geocentric equatorial inertial approximation, km and km/s.
- ECEF position/velocity: Earth-fixed rotating frame, km and km/s.
- Latitude/longitude/altitude: spherical-Earth geographic approximation.

In [ ]:
# =====================================================================
# Stage 5: Kepler solver, ECI-like state, and ECI-to-ECEF transformation
# =====================================================================
J2000 = datetime(2000, 1, 1, 12, tzinfo=timezone.utc)

def solve_kepler(mean_anomaly, eccentricity, tolerance=1e-12):
    M = mean_anomaly % (2 * math.pi)
    E = M if eccentricity < 0.8 else math.pi
    for _ in range(100):
        step = (E - eccentricity * math.sin(E) - M) / (1 - eccentricity * math.cos(E))
        E -= step
        if abs(step) < tolerance:
            return E
    raise RuntimeError("Kepler equation did not converge.")

def gmst_rad(utc_time):
    # IAU-1982-style GMST approximation; UTC is used as an approximation to UT1.
    days = (utc_time - J2000).total_seconds() / 86400.0
    centuries = days / 36525.0
    degrees = (
        280.46061837 + 360.98564736629 * days
        + 0.000387933 * centuries**2 - centuries**3 / 38710000.0
    )
    return math.radians(degrees % 360.0)

def orbital_plane_to_eci(xp, yp, vxp, vyp, inclination, raan, argp):
    # Rotation R3(RAAN) * R1(inclination) * R3(argument of pericentre).
    cO, sO = math.cos(raan), math.sin(raan)
    ci, si = math.cos(inclination), math.sin(inclination)
    cw, sw = math.cos(argp), math.sin(argp)

    r11 = cO*cw - sO*sw*ci
    r12 = -cO*sw - sO*cw*ci
    r21 = sO*cw + cO*sw*ci
    r22 = -sO*sw + cO*cw*ci
    r31 = sw*si
    r32 = cw*si

    return (
        r11*xp + r12*yp, r21*xp + r22*yp, r31*xp + r32*yp,
        r11*vxp + r12*vyp, r21*vxp + r22*vyp, r31*vxp + r32*vyp
    )

def eci_to_ecef(x, y, z, vx, vy, vz, theta):
    c, s = math.cos(theta), math.sin(theta)
    xe = c*x + s*y
    ye = -s*x + c*y
    ze = z

    # Velocity derivative in the rotating ECEF frame:
    # v_ECEF = R * v_ECI - omega_Earth x r_ECEF.
    vxe = c*vx + s*vy + EARTH_ROTATION_RAD_S*ye
    vye = -s*vx + c*vy - EARTH_ROTATION_RAD_S*xe
    vze = vz
    return xe, ye, ze, vxe, vye, vze

def propagate_one(elements_row, time_grid):
    a, e = elements_row["a_km"], elements_row["e"]
    n = elements_row["n_rad_s"]
    inclination, raan, argp = (
        elements_row["i_rad"], elements_row["raan_rad"], elements_row["argp_rad"]
    )
    points = []

    for t in time_grid:
        dt = (t - elements_row["epoch"]).total_seconds()
        M = (elements_row["M0_rad"] + n*dt) % (2*math.pi)
        E = solve_kepler(M, e)
        denom = 1 - e*math.cos(E)

        # Position and velocity in the perifocal/orbital plane.
        xp = a*(math.cos(E) - e)
        yp = a*math.sqrt(1 - e**2)*math.sin(E)
        vxp = -a*n*math.sin(E)/denom
        vyp = a*n*math.sqrt(1 - e**2)*math.cos(E)/denom

        x, y, z, vx, vy, vz = orbital_plane_to_eci(
            xp, yp, vxp, vyp, inclination, raan, argp
        )
        theta = gmst_rad(t)
        xe, ye, ze, vxe, vye, vze = eci_to_ecef(x, y, z, vx, vy, vz, theta)

        radius = math.sqrt(x*x + y*y + z*z)
        speed = math.sqrt(vx*vx + vy*vy + vz*vz)
        lat = math.degrees(math.atan2(ze, math.hypot(xe, ye)))
        lon = math.degrees(math.atan2(ye, xe))
        altitude = math.sqrt(xe*xe + ye*ye + ze*ze) - EARTH_RADIUS_KM

        points.append({
            "time": t,
            "x_eci_km": x, "y_eci_km": y, "z_eci_km": z,
            "vx_eci_km_s": vx, "vy_eci_km_s": vy, "vz_eci_km_s": vz,
            "x_ecef_km": xe, "y_ecef_km": ye, "z_ecef_km": ze,
            "vx_ecef_km_s": vxe, "vy_ecef_km_s": vye, "vz_ecef_km_s": vze,
            "lat_deg": lat, "lon_deg": lon, "alt_km": altitude,
            "radius_km": radius, "speed_km_s": speed
        })
    return points

tracks = [propagate_one(el, times) for el in elements]

# Numerical checks: rotation preserves position norm, and ECI/ECEF inverse rotation recovers position.
for el, pts in zip(elements, tracks):
    for p in (pts[0], pts[len(pts)//2], pts[-1]):
        r_eci = math.sqrt(p["x_eci_km"]**2 + p["y_eci_km"]**2 + p["z_eci_km"]**2)
        r_ecef = math.sqrt(p["x_ecef_km"]**2 + p["y_ecef_km"]**2 + p["z_ecef_km"]**2)
        assert abs(r_eci - r_ecef) < 1e-7
        assert abs(r_eci - p["radius_km"]) < 1e-7

print(f"Propagation complete: {len(elements)} satellites, {n_points} points each.")
for el, pts in zip(elements, tracks):
    p = pts[0]
    print(f"{el['label']}: first ECI position = "
          f"({p['x_eci_km']:.2f}, {p['y_eci_km']:.2f}, {p['z_eci_km']:.2f}) km")

## Stage 6: Orbit summary and physical checks

The summary uses the two-body elements and the propagated scalar speed. The model is educational;
do not interpret these values as precision orbit-determination results.

In [ ]:
# =====================================================================
# Stage 6: orbit summary and checks
# =====================================================================
def orbit_summary(el, points):
    speeds = [p["speed_km_s"] for p in points]
    return {
        "name": el["name"],
        "norad_id": el["norad_id"],
        "period_h": 2*math.pi/el["n_rad_s"]/3600,
        "perigee_km": el["a_km"]*(1-el["e"]) - EARTH_RADIUS_KM,
        "apogee_km": el["a_km"]*(1+el["e"]) - EARTH_RADIUS_KM,
        "e": el["e"],
        "i_deg": math.degrees(el["i_rad"]),
        "speed_min_km_s": min(speeds),
        "speed_max_km_s": max(speeds),
    }

summaries = [orbit_summary(el, pts) for el, pts in zip(elements, tracks)]
for item in summaries:
    print(
        f"{item['name']:<24} NORAD {item['norad_id']:>6} | "
        f"period {item['period_h']:>7.3f} h | "
        f"perigee {item['perigee_km']:>9.1f} km | "
        f"apogee {item['apogee_km']:>9.1f} km | "
        f"speed {item['speed_min_km_s']:.3f}-{item['speed_max_km_s']:.3f} km/s"
    )

# The vis-viva speed and vector speed should agree at every sampled point.
for points in tracks:
    for p in points[::max(1, len(points)//20)]:
        vis_viva = math.sqrt(MU_EARTH_KM3_S2 * (2/p["radius_km"] - 1/next(
            el["a_km"] for el, pts in zip(elements, tracks) if pts is points
        )))
        assert abs(vis_viva - p["speed_km_s"]) < 1e-8

print("Checks passed: ECI/ECEF radius preservation and velocity consistency.")

## Stage 7: ECI orbit plot, ground tracks, and speed profiles

The 3D plot uses the ECI-like Cartesian positions. The ground-track plot uses latitude/longitude
obtained from ECEF coordinates. The speed plot shows inertial orbital speed.

In [ ]:
# =====================================================================
# Stage 7: visualize ECI orbits, ground tracks, and speed
# =====================================================================
COLORS = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple", "tab:brown"]

def load_coastline(path):
    segments, current = [], []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if line.strip():
            lat, lon = map(float, line.split()[:2])
            current.append((lon, lat))
        elif current:
            segments.append(current)
            current = []
    if current:
        segments.append(current)
    return segments

# 3D ECI-like trajectories.
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")
for idx, (el, points) in enumerate(zip(elements, tracks)):
    ax.plot(
        [p["x_eci_km"] for p in points],
        [p["y_eci_km"] for p in points],
        [p["z_eci_km"] for p in points],
        color=COLORS[idx % len(COLORS)], linewidth=1, label=el["label"]
    )
ax.set_xlabel("ECI-like X [km]")
ax.set_ylabel("ECI-like Y [km]")
ax.set_zlabel("ECI-like Z [km]")
ax.set_title("Satellite trajectories in the ECI-like frame")
ax.legend(fontsize=8)
ax.set_box_aspect((1, 1, 1))
fig.tight_layout()
fig.savefig(OUT_DIR / "eci_orbit_tracks_3d.png", dpi=180)
plt.show()

# Ground tracks (ECEF-derived geographic coordinates).
fig, ax = plt.subplots(figsize=(12, 6))
coast_path = IN_DIR / "world_map.txt"
if coast_path.is_file():
    for segment in load_coastline(coast_path):
        lons, lats = zip(*segment)
        ax.plot(lons, lats, color="gray", linewidth=0.6)
for idx, (el, points) in enumerate(zip(elements, tracks)):
    ax.scatter(
        [p["lon_deg"] for p in points], [p["lat_deg"] for p in points],
        s=3, color=COLORS[idx % len(COLORS)], label=el["label"]
    )
ax.set(xlim=(-180, 180), ylim=(-90, 90), xlabel="Longitude [deg]",
       ylabel="Geocentric latitude [deg]", title="Ground tracks from ECEF coordinates")
ax.grid(linestyle=":", alpha=0.5)
ax.legend(loc="upper right", fontsize=8, markerscale=3)
fig.tight_layout()
fig.savefig(OUT_DIR / "ground_tracks.png", dpi=180)
plt.show()

# Orbital speed time series.
fig, axes = plt.subplots(len(tracks), 1, figsize=(12, 2.5*len(tracks)), sharex=True, squeeze=False)
for idx, (el, points) in enumerate(zip(elements, tracks)):
    ax = axes[idx, 0]
    ax.plot([p["time"] for p in points], [p["speed_km_s"] for p in points],
            color=COLORS[idx % len(COLORS)], linewidth=1.5)
    ax.set_title(el["label"], fontsize=9, loc="left")
    ax.set_ylabel("Speed [km/s]")
    ax.grid(linestyle=":", alpha=0.5)
axes[-1, 0].set_xlabel("Time (UTC)")
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(OUT_DIR / "velocity_timeseries.png", dpi=180)
plt.show()

## Stage 8: Export ECI/ECEF data, KML, and report

The CSV files preserve the actual Cartesian coordinates and velocity components. KML uses
longitude, latitude, and altitude because that is the coordinate representation expected by KML;
the KML file is not a raw ECEF Cartesian file.

In [ ]:
# =====================================================================
# Stage 8A: export full ECI-like and ECEF state vectors to CSV
# =====================================================================
CSV_FIELDS = [
    "time_utc", "satellite", "norad_id",
    "x_km", "y_km", "z_km", "vx_km_s", "vy_km_s", "vz_km_s"
]

def export_state_csv(filename, frame):
    path = OUT_DIR / filename
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=CSV_FIELDS)
        writer.writeheader()
        for el, points in zip(elements, tracks):
            for p in points:
                if frame == "ECI":
                    keys = ("x_eci_km", "y_eci_km", "z_eci_km",
                            "vx_eci_km_s", "vy_eci_km_s", "vz_eci_km_s")
                elif frame == "ECEF":
                    keys = ("x_ecef_km", "y_ecef_km", "z_ecef_km",
                            "vx_ecef_km_s", "vy_ecef_km_s", "vz_ecef_km_s")
                else:
                    raise ValueError("frame must be ECI or ECEF")
                writer.writerow({
                    "time_utc": p["time"].isoformat(),
                    "satellite": el["name"],
                    "norad_id": el["norad_id"],
                    "x_km": p[keys[0]], "y_km": p[keys[1]], "z_km": p[keys[2]],
                    "vx_km_s": p[keys[3]], "vy_km_s": p[keys[4]], "vz_km_s": p[keys[5]]
                })
    print(f"Written: {path.name}")
    return path

eci_csv = export_state_csv("orbit_tracks_eci_like.csv", "ECI")
ecef_csv = export_state_csv("orbit_tracks_ecef.csv", "ECEF")

In [ ]:
# =====================================================================
# Stage 8B: create KML files for Google Earth
# =====================================================================
KML_NS = "http://www.opengis.net/kml/2.2"
GX_NS = "http://www.google.com/kml/ext/2.2"
ET.register_namespace("", KML_NS)
ET.register_namespace("gx", GX_NS)

def kml(tag):
    return f"{{{KML_NS}}}{tag}"

def gx(tag):
    return f"{{{GX_NS}}}{tag}"

def kml_color(index):
    rgb = to_hex(COLORS[index % len(COLORS)]).lstrip("#")
    return "ff" + rgb[4:6] + rgb[2:4] + rgb[0:2]

def new_kml_document(title, description):
    root = ET.Element(kml("kml"))
    doc = ET.SubElement(root, kml("Document"))
    ET.SubElement(doc, kml("name")).text = title
    ET.SubElement(doc, kml("description")).text = description
    for idx in range(len(elements)):
        style = ET.SubElement(doc, kml("Style"), id=f"sat{idx}")
        line_style = ET.SubElement(style, kml("LineStyle"))
        ET.SubElement(line_style, kml("color")).text = kml_color(idx)
        ET.SubElement(line_style, kml("width")).text = "2"
    return root, doc

def add_placemark(doc, idx, name):
    placemark = ET.SubElement(doc, kml("Placemark"))
    ET.SubElement(placemark, kml("name")).text = name
    ET.SubElement(placemark, kml("styleUrl")).text = f"#sat{idx}"
    return placemark

def split_at_dateline(points):
    segments = [[points[0]]]
    for previous, point in zip(points, points[1:]):
        if abs(point["lon_deg"] - previous["lon_deg"]) > 180:
            segments.append([])
        segments[-1].append(point)
    return [segment for segment in segments if segment]

# Static ground-track/orbit KML: coordinates are lon,lat,altitude in metres.
root, doc = new_kml_document(
    "Satellite tracks (ECEF-derived geographic coordinates)",
    "Educational two-body model; spherical Earth; approximate GMST rotation."
)
for idx, (el, points) in enumerate(zip(elements, tracks)):
    line = ET.SubElement(add_placemark(doc, idx, el["label"]), kml("LineString"))
    ET.SubElement(line, kml("altitudeMode")).text = "absolute"
    ET.SubElement(line, kml("coordinates")).text = " ".join(
        f"{p['lon_deg']:.6f},{p['lat_deg']:.6f},{p['alt_km']*1000:.1f}" for p in points
    )
ET.ElementTree(root).write(OUT_DIR / "orbit_tracks_google_earth.kml",
                           encoding="utf-8", xml_declaration=True)

# Time animation, sampled at approximately one-minute intervals.
animation_step = max(1, round(60 / STEP_SECONDS))
root, doc = new_kml_document(
    "Satellite ground-track animation",
    "Time-tagged geographic coordinates for Google Earth."
)
for idx, (el, points) in enumerate(zip(elements, tracks)):
    sampled = points[::animation_step]
    if points[-1] is not sampled[-1]:
        sampled.append(points[-1])
    multi = ET.SubElement(add_placemark(doc, idx, el["label"]), gx("MultiTrack"))
    for segment in split_at_dateline(sampled):
        track = ET.SubElement(multi, gx("Track"))
        ET.SubElement(track, kml("altitudeMode")).text = "clampToGround"
        for p in segment:
            ET.SubElement(track, kml("when")).text = p["time"].strftime("%Y-%m-%dT%H:%M:%SZ")
        for p in segment:
            ET.SubElement(track, gx("coord")).text = (
                f"{p['lon_deg']:.6f} {p['lat_deg']:.6f} 0"
            )
ET.ElementTree(root).write(OUT_DIR / "orbit_animation.kml",
                           encoding="utf-8", xml_declaration=True)

print("KML exports written.")

In [ ]:
# =====================================================================
# Stage 8C: write report and package all outputs
# =====================================================================
report_lines = [
    "Satellite Orbit Report — ECI-like and ECEF outputs",
    "",
    f"Data source: {source_note}",
    f"Window UTC: {start:%Y-%m-%d %H:%M:%S} -> {end:%Y-%m-%d %H:%M:%S}",
    f"Sampling: {STEP_SECONDS} s; {n_points} samples per satellite",
    "",
    "MODEL:",
    "Two-body Kepler propagation using GP/OMM mean elements.",
    "ECI-like frame: geocentric equatorial inertial approximation.",
    "ECEF: GMST rotation; UTC approximates UT1.",
    "Spherical Earth; no SGP4, J2, drag, precession/nutation, or polar motion.",
    "Educational approximation; not precision orbit prediction.",
    "",
    "ORBIT SUMMARY"
]
for item in summaries:
    report_lines.append(
        f"{item['name']} | NORAD {item['norad_id']} | period {item['period_h']:.4f} h | "
        f"perigee {item['perigee_km']:.2f} km | apogee {item['apogee_km']:.2f} km | "
        f"speed {item['speed_min_km_s']:.4f}-{item['speed_max_km_s']:.4f} km/s"
    )
report_lines += ["", "ELEMENTS AT EPOCH", *elements_table(elements)]
report_path = OUT_DIR / "orbit_report.txt"
report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")

deliverables = [
    "orbit_tracks_eci_like.csv",
    "orbit_tracks_ecef.csv",
    "eci_orbit_tracks_3d.png",
    "ground_tracks.png",
    "velocity_timeseries.png",
    "orbit_tracks_google_earth.kml",
    "orbit_animation.kml",
    "orbit_report.txt",
]
zip_path = OUT_DIR / "satellite_orbit_outputs_corrected.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for filename in deliverables:
        file_path = OUT_DIR / filename
        if not file_path.is_file():
            raise FileNotFoundError(f"Expected output was not created: {file_path}")
        archive.write(file_path, arcname=filename)

print("Final output files:")
for filename in deliverables:
    print(f"  {filename}")
print(f"\nPackaged archive: {zip_path}")
print("Notebook processing completed only if this cell reaches the final print statements.")